# 02. s02_plateau_pathways

Step 2. Plateau ratings, climate-pathway decomposition with block bootstrap, rating-change tests (Tables 2, B1; Figs 4-6 inputs).

Run the notebooks in numerical order (each reads outputs of earlier steps from `results/`).

In [1]:
import os, sys
try:
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
except NameError:  # running inside a Jupyter notebook
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from config import *  # sets paths; working directory becomes results/

In [2]:
import pandas as pd,numpy as np,json,warnings; warnings.filterwarnings('ignore')
import statsmodels.api as sm
from scipy.stats import kendalltau, spearmanr
EN={'澜沧江':'Lancang','疏勒河':'Shule','塔里木河':'Tarim','怒江':'Nu','雅鲁藏布江':'Yarlung Tsangpo','印度河':'Indus'}
STN={'香达':'Xiangda','旧州':'Jiuzhou','昌马堡':'Changmabao','潘家庄':'Panjiazhuang','卡群':'Kaqun','同古孜洛克':'Tongguziluoke','新大河':'Xindahe','焉耆':'Yanqi','阿拉尔':'Alar','道街坝':'Daojieba','拉萨':'Lhasa','奴下':'Nuxia'}
x=pd.read_excel(XLS_TP,sheet_name=None,header=None)
rng=np.random.default_rng(5); rows=[]; series={}
def mbb(n,L):
  return np.concatenate([np.arange(s,s+L) for s in rng.integers(0,n-L+1,int(np.ceil(n/L)))])[:n]
for sh,df in x.items():
  for c in range(df.shape[1]):
    if not str(df.iat[0,c]).startswith('站点名'): continue
    st=STN.get(df.iat[0,c+1],df.iat[0,c+1]); d=df.iloc[2:,c:c+5].apply(pd.to_numeric,errors='coerce'); d.columns=['yr','Q','S','T','P']; d=d.dropna(); d=d[(d.Q>0)&(d.S>0)].reset_index(drop=True)
    n=len(d); lS=np.log(d.S.values); lQ=np.log(d.Q.values); T=d['T'].values-d['T'].mean(); lP=np.log(d.P.values/d.P.mean()); yr=d.yr.values
    series[st]=d.assign(river=EN[sh],station=st)
    def est(idx):
      XS=sm.add_constant(np.c_[lQ[idx]-lQ.mean(),T[idx],lP[idx]]); bs=np.linalg.lstsq(XS,lS[idx],rcond=None)[0]
      XQ=sm.add_constant(np.c_[T[idx],lP[idx]]); bq=np.linalg.lstsq(XQ,lQ[idx],rcond=None)[0]
      XT=sm.add_constant(np.c_[T[idx],lP[idx]]); bt=np.linalg.lstsq(XT,lS[idx],rcond=None)[0]
      return dict(b=bs[1],cT=bs[2],cP=bs[3],qT=bq[1],qP=bq[2],totT=bt[1],totP=bt[2])
    base=est(np.arange(n)); B=[est(mbb(n,3)) for _ in range(2000)]
    def ci(k): v=np.array([e[k] for e in B]); return np.percentile(v,[2.5,97.5])
    hydT=np.array([e['b']*e['qT'] for e in B]); hydP=np.array([e['b']*e['qP'] for e in B])
    r=dict(river=EN[sh],station=st,period=f'{int(yr.min())}–{int(yr.max())}',n=n)
    for k in ['b','cT','cP','qT','qP','totT','totP']:
      lo,hi=ci(k); r[k]=round(base[k],3); r[k+'_lo']=round(lo,3); r[k+'_hi']=round(hi,3)
    r['hydT']=round(base['b']*base['qT'],3); r['hydT_lo'],r['hydT_hi']=np.round(np.percentile(hydT,[2.5,97.5]),3)
    r['hydP']=round(base['b']*base['qP'],3); r['hydP_lo'],r['hydP_hi']=np.round(np.percentile(hydP,[2.5,97.5]),3)
    # standardized runoff sensitivity: melt vs rain index
    sT=base['qT']*np.std(T); sP=base['qP']*np.std(lP); r['melt_index']=round(sT/(abs(sT)+abs(sP)+1e-9),2)
    # rating change: interaction with late period
    late=(np.arange(n)>=n//2).astype(float); Xi=sm.add_constant(np.c_[lQ-lQ.mean(),late,(lQ-lQ.mean())*late])
    oi=sm.OLS(lS,Xi).fit(cov_type='HAC',cov_kwds={'maxlags':2}); r['db']=round(oi.params[3],2); r['p_db']=round(oi.pvalues[3],3)
    r['b_early']=round(oi.params[1],2); r['b_late']=round(oi.params[1]+oi.params[3],2); r['da']=round(oi.params[2],2); r['p_da']=round(oi.pvalues[2],3)
    o0=sm.OLS(lS,sm.add_constant(lQ)).fit(); r['R2_rating']=round(o0.rsquared,2)
    tt,pp=kendalltau(yr,d['T']); r['T_trend_p']=round(pp,3); r['T_sen_dec']=round(10*np.median([(d['T'][j]-d['T'][i])/(yr[j]-yr[i]) for i in range(n) for j in range(i+1,n) if yr[j]!=yr[i]]),3)
    tq,pq=kendalltau(yr,d.Q); r['Q_trend_tau']=round(tq,2); r['Q_trend_p']=round(pq,3)
    ts,ps=kendalltau(yr,d.S); r['S_trend_tau']=round(ts,2); r['S_trend_p']=round(ps,3)
    rows.append(r)
A=pd.DataFrame(rows); A.to_csv('tp_station_results.csv',index=False)
pd.set_option('display.width',300); print(A[['river','station','n','R2_rating','b','b_lo','b_hi','qT','qT_lo','qT_hi','qP','cT','cT_lo','cT_hi','cP','cP_lo','cP_hi','hydT','hydT_lo','hydT_hi','totT','melt_index','b_early','b_late','p_db','T_sen_dec','T_trend_p']].to_string())
good=A[A.R2_rating>=0.3]
S={}
S['n_good']=len(good); S['n_cT_pos_sig']=int(((good.cT_lo>0)).sum()); S['n_cT_neg_sig']=int(((good.cT_hi<0)).sum()); S['n_cT_ns']=int(((good.cT_lo<=0)&(good.cT_hi>=0)).sum())
S['n_hydT_pos_sig']=int((good.hydT_lo>0).sum()); S['n_qT_pos']=int((good.qT_lo>0).sum()); S['n_qP_pos']=int((good.qP_lo>0).sum())
S['median_b_good']=float(good.b.median()); S['b_range_good']=[float(good.b.min()),float(good.b.max())]
S['n_db_sig']=int((A.p_db<0.05).sum()); S['db_sig']=A[A.p_db<0.05][['station','b_early','b_late','p_db']].to_dict('records')
S['n_T_sig']=int((A.T_trend_p<0.05).sum()); S['T_sen_range']=[float(A.T_sen_dec.min()),float(A.T_sen_dec.max())]
melt=good[good.melt_index>0.5]; rain=good[good.melt_index<=0.5]
S['melt_st']=melt.station.tolist(); S['rain_st']=rain.station.tolist()
rho,p=spearmanr(good.melt_index,good.hydT); S['rho_melt_hydT']=[round(rho,2),round(p,3)]
json.dump(S,open('tp_summary.json','w'),indent=1,default=float); print(json.dumps(S,indent=1,default=float))
pd.concat(series.values()).to_csv('tp_series.csv',index=False)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
